# Compare Giano with linear interpolation
Hide all three measured variables at one station at a time. Start at midnight UTC on 1 March, 1 June and 1 September 2024. Use 6-hour and 168-hour outages at each station. The frozen checkpoints and both methods see the same damaged input. Linear interpolation can use a visible temperature on either side of a gap.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from giano.data import load_input

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sample = root / "examples" / "noaa-2024"
torch.set_num_threads(2)
ds = load_input(sample / "observations.csv", sample / "stations.csv")

In [ ]:
import sys

sys.path.insert(0, str(root / "examples"))

In [ ]:
from evaluate_temperature import compare, scores

rows = compare(ds)
scores(rows).round(3)

## Inspect the denominator
A temperature contributes to MAE when the measured target and both estimates exist. Count scheduled, observed and paired hours for each duration. Each scored hour has equal weight in the summary.

In [ ]:
rows.assign(paired=rows[["observed", "giano", "linear"]].notna().all(axis=1)).groupby(
    "hours"
).agg(
    scheduled_hours=("time", "size"),
    measured_hours=("observed", "count"),
    paired_hours=("paired", "sum"),
)

## Inspect one outage
Plot the first seven-day outage at Locarno-Monti. Here the observed temperatures were hidden from both methods until scoring. An aggregate score alone cannot show whether a method follows the daily cycle.

In [ ]:
episode = rows.loc[
    (rows.start == "2024-03-01")
    & (rows.hours == 168)
    & (rows.station == "067600-99999")
]
ax = episode.set_index("time")[["observed", "giano", "linear"]].plot(
    ylabel="Temperature (°C)", xlabel="Time (UTC)", title="Locarno-Monti, March 2024"
)
plt.show()

## Compare historical Trentino results by outage regime
The Swiss check covers one variable in a three-station network. The 2020 Trentino evaluation includes all six variables, five durations and three regimes. Inspect the number of paired observations alongside the error. Use the included 90-row table with both station options enabled, or rerun `giano-evaluate --local-context --station-reference` and read your own output from `runs/evaluation/scores.csv`.

In [ ]:
import pandas as pd

historical = root / "runs/evaluation/scores.csv"
if not historical.is_file():
    historical = root / "examples/trentino/station-context-scores.csv"
by_case = pd.read_csv(historical)
print(historical)
by_case.query("variable == 'humidity' and hours == 720")[
    ["regime", "measured", "paired", "giano_mae", "point_mae", "physical_mae"]
]

Each humidity MAE is in percentage points. Here `point_mae` is the error of the neural ensemble before the external humidity correction. `physical_mae` is the error of the physical reference.